<a href="https://colab.research.google.com/github/arshdeepbangar/AAI2025/blob/main/Arsh_Bangar_Exercise_1_Prompt_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 - Prompt Chaining for a Customer Support AI

**Goal:** Build a multi-step customer-service flow where each step uses output from the previous step.

**Tools used:** ChatGPT for testing the prompts and Google Colab/Python for showing the chain and outputs.

### Chain design
1. Classify the issue and identify missing information.
2. Ask only for the missing information.
3. Propose a solution using the earlier classification and the new customer details.
4. Apply an escalation rule and produce the final response.

## Test customer message

> Hi, I ordered wireless headphones three days ago and paid for two-day shipping, but the tracking page still says "label created." I need them before Friday. Can you help?

In [1]:
import json
customer_message = ("Hi, I ordered wireless headphones three days ago and paid for two-day shipping, "
                    "but the tracking page still says 'label created.' I need them before Friday. Can you help?")
print(customer_message)

Hi, I ordered wireless headphones three days ago and paid for two-day shipping, but the tracking page still says 'label created.' I need them before Friday. Can you help?


## Step 1 Prompt - Classify the issue

You are a customer support triage assistant. Read the customer message and return exactly five fields: **Category, Urgency, Known information, Missing information, Next action**.

Constraints: use a calm professional tone; do not invent order details; do not request passwords, full payment card numbers, or other sensitive information; request only information needed to continue.

Customer message: `{customer_message}`

In [2]:
step1_output = {
    "Category": "Shipping delay / package not scanned",
    "Urgency": "High because the customer paid for two-day shipping and has a Friday deadline",
    "Known information": ["Wireless headphones ordered three days ago", "Two-day shipping paid", "Tracking says label created", "Needed before Friday"],
    "Missing information": ["Order number", "Email address or ZIP code used for the order"],
    "Next action": "Ask for the order number and one verification detail"
}
print(json.dumps(step1_output, indent=2))

{
  "Category": "Shipping delay / package not scanned",
  "Urgency": "High because the customer paid for two-day shipping and has a Friday deadline",
  "Known information": [
    "Wireless headphones ordered three days ago",
    "Two-day shipping paid",
    "Tracking says label created",
    "Needed before Friday"
  ],
  "Missing information": [
    "Order number",
    "Email address or ZIP code used for the order"
  ],
  "Next action": "Ask for the order number and one verification detail"
}


## Step 2 Prompt - Gather only missing information

This prompt uses **Step 1 output** as input.

You are a customer support representative. Based only on the triage result below, write a short message asking for the missing information.

Constraints: maximum 3 sentences; be polite and reassuring; ask only for the order number and either the email address or ZIP code used for the order; do not ask for a password, Social Security number, or payment card information; do not blame the customer or carrier.

Triage result: `{step1_output}`

In [3]:
step2_prompt = f"Triage result: {step1_output}"
step2_output = ("I can help check the shipment for you. Please send your order number and either the email address or ZIP code used for the order. "
                "Please do not send any password or payment card information.")
print(step2_output)

I can help check the shipment for you. Please send your order number and either the email address or ZIP code used for the order. Please do not send any password or payment card information.


### Customer reply used for testing

> Order number A10482. ZIP code 10001.

In [4]:
customer_reply = {"order_number":"A10482", "zip_code":"10001"}
print(customer_reply)

{'order_number': 'A10482', 'zip_code': '10001'}


## Step 3 Prompt - Propose a solution

This prompt uses **Step 1 classification**, **Step 2 customer details**, and a test order-status lookup.

Test order status: no carrier scan for more than 48 hours. The order is eligible for a free overnight replacement or a refund of the shipping charge.

You are a customer support resolution assistant. Use the triage result, verified customer details, and order status to propose a solution.

Constraints: give no more than 2 options; put the fastest option first; do not promise an unconfirmed delivery date; do not blame the customer or carrier; end by asking which option the customer wants.

In [ ]:
order_status = {"scan_status":"No carrier scan for more than 48 hours", "replacement_eligible":True, "replacement_shipping":"Overnight at no charge", "shipping_refund_eligible":True}
step3_output = ("The fastest option is a free overnight replacement because the original package has not been scanned for more than 48 hours. "
                "If you prefer to keep waiting for the original package, we can refund the two-day shipping charge instead. Which option would you like?")
print(step3_output)

The fastest option is a free overnight replacement because the original package has not been scanned for more than 48 hours. If you prefer to keep waiting for the original package, we can refund the two-day shipping charge instead. Which option would you like?


## Step 4 Prompt - Apply the escalation rule

**Escalation rule:** Escalate to a human agent when an expedited package has no carrier scan for more than 48 hours **and** the customer has a time-sensitive deadline within two days.

Review the triage result and proposed solution. Apply the escalation rule exactly. Output exactly: **Escalate: Yes/No, Reason, Final customer message**.

The final customer message must be under 80 words, polite, and must not guarantee an unconfirmed delivery date.

In [ ]:
step4_output = {
    "Escalate":"Yes",
    "Reason":"The package was expedited, has had no carrier scan for more than 48 hours, and the customer has a time-sensitive deadline.",
    "Final customer message":"Your order qualifies for escalation because the expedited package has not received a carrier scan for more than 48 hours and you have a Friday deadline. I can have a support agent review a free overnight replacement, or you can request a refund of the two-day shipping charge."
}
print(json.dumps(step4_output, indent=2))

{
  "Escalate": "Yes",
  "Reason": "The package was expedited, has had no carrier scan for more than 48 hours, and the customer has a time-sensitive deadline.",
  "Final customer message": "Your order qualifies for escalation because the expedited package has not received a carrier scan for more than 48 hours and you have a Friday deadline. I can have a support agent review a free overnight replacement, or you can request a refund of the two-day shipping charge."
}


## Testing and iteration

**First version of Step 2:** “Ask the customer for their information so we can look up the order.”

**Problem:** It was too vague and could cause the AI to request unnecessary or sensitive information.

**Improvement:** The final Step 2 prompt names the exact fields needed, limits the response to 3 sentences, and says not to request passwords or payment information.

## Final successful output

The chain classified the issue, asked only for needed information, proposed two valid options, and triggered escalation using the stated rule. Each later step uses information produced earlier in the chain.